# Create Medical Research Future Fund (MRFF) Awards

**MRFF** (funder_id `4906014721` — Path B, **MISSING from openalex.common.funder**;
display_name from the live API (DOI from Crossref/existing awards), registry-gap flagged — IAMHRF priority-40,
priority `351`). The funder's own bulk Excel export (health.gov.au "MRFF grant
recipients", manual browser download — Akamai + .gov.au geo-wall block scripts;
see scripts/local/mrff_to_s3.py). **1,836 grants**, native `Grant ID`, 100%
title/institution/AUD-amount/dates, **PI 95%** (Chief Investigator A), scheme,
research area. The richest dedicated funder in the campaign.

Parquet: `s3://openalex-ingest/awards/mrff/mrff_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mrff_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/mrff/mrff_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.mrff_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.mrff_raw LIMIT 5;

## Step 2: Create MRFF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mrff_awards
USING delta
AS
WITH
mrff_funder AS (
    -- F4906014721 is MISSING from openalex.common.funder (registry gap, flagged);
    -- display_name from api.openalex.org/funders/F4906014721 (2026-06-15; ROR + DOI both null at the funder endpoint); the 10.13039/501100025520 DOI is taken from Crossref / existing live MRFF award rows, which carry it. Safe downstream: CreateAwards/CreateAwardsAPI preserve this embedded struct and do NOT re-join common.funder.
    SELECT CAST(4906014721 AS BIGINT) as funder_id,
           'Medical Research Future Fund' as display_name,
           CAST(NULL AS STRING) as ror_id,
           '10.13039/501100025520' as doi
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        COALESCE(NULLIF(TRIM(g.title), ''), g.scheme, CONCAT('MRFF grant: ', g.institution), g.funder_award_id) as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        TRY_CAST(g.amount AS DECIMAL(18,2)) as amount,
        'AUD' as currency,
        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name, f.ror_id, f.doi
        ) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'mrff' as provenance,
        TRY_TO_DATE(g.start_date_raw, 'yyyy-MM-dd') as start_date,
        TRY_TO_DATE(g.end_date_raw, 'yyyy-MM-dd') as end_date,
        YEAR(TRY_TO_DATE(g.start_date_raw, 'yyyy-MM-dd')) as start_year,
        YEAR(TRY_TO_DATE(g.end_date_raw, 'yyyy-MM-dd')) as end_year,
        CASE
            WHEN g.pi_family IS NOT NULL THEN
                struct(
                    g.pi_given as given_name,
                    g.pi_family as family_name,
                    CAST(NULL AS STRING) as orcid,
                    CAST(NULL AS DATE) as role_start,
                    struct(
                        g.institution as name,
                        'Australia' as country,
                        CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
                    ) as affiliation
                )
            ELSE NULL
        END as lead_investigator,
        CAST(NULL AS STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.mrff_raw g
    CROSS JOIN mrff_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mrff' AND priority = 351;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    351 as priority
FROM openalex.awards.mrff_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, COUNT(DISTINCT funder_award_id) uniq_award,
  SUM(CASE WHEN display_name IS NULL OR LENGTH(TRIM(display_name))=0 THEN 1 ELSE 0 END) blank,
  COUNT(amount) has_amount, COUNT(lead_investigator) has_pi, COUNT(start_date) has_start,
  ROUND(SUM(amount)/1000000000,2) total_billion_aud,
  MIN(start_year) min_yr, MAX(start_year) max_yr
FROM openalex.awards.mrff_awards;

In [ ]:
%sql
SELECT lead_investigator.affiliation.name inst, COUNT(*) cnt, ROUND(SUM(amount)/1000000,0) musd
FROM openalex.awards.mrff_awards WHERE lead_investigator IS NOT NULL
GROUP BY 1 ORDER BY 2 DESC LIMIT 15;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw WHERE provenance='mrff' AND priority=351;